# R4 full-development Lenta comparison (CPU)
Attach the private Dataset built from `kaggle/build_bundle.py`. This is exploratory because Lenta propensity is assumed. Run cells in order; do not edit the protocol or use confirmation labels.

In [ ]:
import hashlib
import importlib.metadata
import json
import subprocess
import sys
from pathlib import Path

roots = [p.parent for p in Path('/kaggle/input').rglob('run_learner.py') if (p.parent / 'INPUT_MANIFEST.json').is_file()]
if len(roots) != 1: raise RuntimeError(f'Attach exactly one R4 kit; found {roots}')
root = roots[0]
expected = json.loads((root / 'INPUT_MANIFEST.json').read_text())
def sha_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''): digest.update(block)
    return digest.hexdigest()
for name, want in expected.items():
    got = sha_file(root / name)
    if got != want: raise RuntimeError(f'input SHA mismatch: {name}')
print('Input SHA-256 checks passed:', list(expected))

In [ ]:
pins = {'numpy':'2.4.6','pandas':'2.3.3','scipy':'1.17.1','scikit-learn':'1.7.2','catboost':'1.2.10'}
def current_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
installed = {name: current_version(name) for name in pins}
if any(installed[name] != want for name, want in pins.items()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(root / 'requirements.txt')], check=True)
wheel = next(root.glob('telcochurn_decision_system-0.1.0-*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', str(wheel)], check=True)
print('Wheel installed; notebook subprocesses will import the pinned source.')

In [ ]:
base = [sys.executable, str(root / 'run_learner.py')]
subprocess.run(base + ['preflight', '--study', 'full-lenta', '--input-root', str(root), '--output', '/kaggle/working/r4_full_lenta_preflight'], check=True)
print(json.loads(Path('/kaggle/working/r4_full_lenta_preflight/RUN_MANIFEST.json').read_text())['status'])

In [ ]:
subprocess.run(base + ['run', '--study', 'full-lenta', '--input-root', str(root), '--output', '/kaggle/working/r4_full_lenta'], check=True)
manifest = json.loads(Path('/kaggle/working/r4_full_lenta/RUN_MANIFEST.json').read_text())
if manifest['status'] != 'COMPLETE': raise RuntimeError('study did not complete')
if manifest['domains']['lenta_sms']['n_development'] != 549867: raise RuntimeError('wrong development count')
print({k: manifest[k] for k in ('status', 'finished_utc', 'elapsed_seconds')})
print('Result SHA-256:', manifest['domains']['lenta_sms']['result_sha256'])